In [ ]:
#Figure S8A & S8B_Tail pinch-evoked DA response (z-score baseline -60~-10 s)

import os
import csv
import numpy as np
import pandas as pd
import tdt
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams["svg.fonttype"] = "none"   # SVG 텍스트 편집 가능하게 저장

# ──────────────────────────────────────────────────────────────────────────────
# 0) 사용자 설정
# ──────────────────────────────────────────────────────────────────────────────
save_dir        = "/path/to/your/TDT_data"

GRAB_CH         = "_465A"
ISO_CH          = "_405A"
DOWNSAMPLE      = 10

# 시간 설정 (단위: 초, tail pinch onset = 0)
BASELINE_START  = -60      # z-score baseline 및 그룹 분류 기준 구간 (양끝 포함)
BASELINE_END    = -10
EARLY_START     = 0        # 그룹 분류용 tail pinch 구간 [0, 3)
EARLY_END       = 3
TRACE_START     = -60      # mean trace 구간
TRACE_END       = 120
TRACE_STEP      = 0.05     # mean trace 공통 시간축 간격 (50 ms)
SMOOTH_S        = 0.5      # mean trace 표시용 이동평균 (초). 분류/peak에는 사용 안 함

# 그룹별 peak 구간 (양끝 포함)
PEAK_WINDOWS = {
    "Increase": [("max", 0, 30)],
    "Decrease": [("min", 0, 3), ("max", 3, 30)],
}

# Pinch 1st / 2nd 경로 (같은 마우스끼리 같은 순서로 입력)
pinch_1st_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]
pinch_2nd_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 1) 핵심 함수
# ──────────────────────────────────────────────────────────────────────────────
def block_mean(x, n):
    """n개 샘플씩 겹치지 않게 평균 (다운샘플링)"""
    m = len(x) // n * n
    return x[:m].reshape(-1, n).mean(axis=1)


def find_pinch_onset(blk):
    """Note epoc의 2000번대 코드(2000~2999) 중 시간상 마지막 onset을 tail pinch onset으로 사용.
    (한 녹화에 2003, 2004가 둘 다 있는 경우가 있어 코드를 고정하지 않음)"""
    codes  = np.asarray(blk.epocs["Note"].data, float).ravel()
    onsets = np.asarray(blk.epocs["Note"].onset, float).ravel()
    n = min(len(codes), len(onsets))
    codes, onsets = codes[:n], onsets[:n]
    ok = np.isfinite(codes) & np.isfinite(onsets) & (codes >= 2000) & (codes < 3000)
    if not ok.any():
        raise KeyError("2000번대 Note 코드를 찾을 수 없습니다.")
    i = np.flatnonzero(ok)[np.argmax(onsets[ok])]
    return onsets[i], int(codes[i])


def load_zscore(block_path):
    """다운샘플 → 405 nm OLS fit → ΔF/F → z-score (baseline -60~-10 s).
    반환 시간축은 tail pinch onset = 0 기준."""
    blk = tdt.read_block(block_path)
    F465 = blk.streams[GRAB_CH].data.astype(float)
    F405 = blk.streams[ISO_CH].data.astype(float)
    fs   = blk.streams[GRAB_CH].fs
    n    = min(len(F465), len(F405))

    F465_ds = block_mean(F465[:n], DOWNSAMPLE)
    F405_ds = block_mean(F405[:n], DOWNSAMPLE)
    time_ds = block_mean(np.arange(n) / fs, DOWNSAMPLE)

    p_coef = np.polyfit(F405_ds, F465_ds, 1)
    fit    = p_coef[0] * F405_ds + p_coef[1]
    dFF    = (F465_ds - fit) / fit

    onset, code = find_pinch_onset(blk)
    t = time_ds - onset

    base = dFF[(t >= BASELINE_START) & (t <= BASELINE_END)]
    z = (dFF - base.mean()) / base.std(ddof=1)
    return t, z, onset, code

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 2) 마우스별 처리: 그룹 분류(1st 기준) + peak + mean trace용 보간
# ──────────────────────────────────────────────────────────────────────────────
TRACE_TIME = np.arange(TRACE_START, TRACE_END + TRACE_STEP / 2, TRACE_STEP)

results = []
for p1, p2 in zip(pinch_1st_paths, pinch_2nd_paths):
    mouse = os.path.basename(os.path.dirname(p1))
    print(f"처리 중: {mouse}")
    data = {"1st": load_zscore(p1), "2nd": load_zscore(p2)}

    # 그룹 분류: ΔZ = mean Z(0~3 s) - mean Z(-60~-10 s), 1st 녹화 기준
    t1, z1 = data["1st"][:2]
    delta = (z1[(t1 >= EARLY_START) & (t1 < EARLY_END)].mean()
             - z1[(t1 >= BASELINE_START) & (t1 <= BASELINE_END)].mean())
    group = "Increase" if delta > 0 else "Decrease"

    rec = {"MouseID": mouse, "Group": group, "DeltaZ": delta, "peaks": {}, "traces": {}}
    for s, (t, z, onset, code) in data.items():
        rec[f"Code_{s}"], rec[f"Onset_{s}"] = code, onset
        rec["traces"][s] = np.interp(TRACE_TIME, t, z, left=np.nan, right=np.nan)

    # Peak: 다운샘플된 원 샘플에서 추출 (보간 trace 사용 안 함)
    for kind, w0, w1 in PEAK_WINDOWS[group]:
        vals = []
        for s in ("1st", "2nd"):
            t, z = data[s][:2]
            seg = z[(t >= w0) & (t <= w1)]
            vals.append(seg.max() if kind == "max" else seg.min())
        rec["peaks"][f"{kind}_{w0}_{w1}s"] = vals
    results.append(rec)

for g in ("Increase", "Decrease"):
    members = [r["MouseID"] for r in results if r["Group"] == g]
    print(f"{g} (n={len(members)}): {', '.join(members)}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 3) CSV 저장
# ──────────────────────────────────────────────────────────────────────────────
# (A) Figure S8A: 마우스별 ΔZ
out_A = os.path.join(save_dir, "FigS8A_deltaZ.csv")
with open(out_A, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["MouseID", "Group", "DeltaZ_0-3s_minus_-60to-10s", "PinchCode_1st", "PinchOnset_1st_s"])
    for r in sorted(results, key=lambda r: r["DeltaZ"]):
        writer.writerow([r["MouseID"], r["Group"], r["DeltaZ"], r["Code_1st"], r["Onset_1st"]])

# (B-right) Figure S8B: 마우스별 peak (1st, 2nd, 2nd-1st = 그래프의 normalized 값)
out_peak = os.path.join(save_dir, "FigS8B_peaks.csv")
with open(out_peak, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Group", "Metric", "MouseID", "Peak_1st", "Peak_2nd", "Normalized_2nd_minus_1st"])
    for g in ("Increase", "Decrease"):
        for r in [r for r in results if r["Group"] == g]:
            for metric, (v1, v2) in r["peaks"].items():
                writer.writerow([g, metric, r["MouseID"], v1, v2, v2 - v1])

# (B-right) Figure S8B: paired t-test (2nd vs 1st)
out_stat = os.path.join(save_dir, "FigS8B_paired_ttest.csv")
with open(out_stat, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Group", "Metric", "n", "Mean_1st", "SEM_1st", "Mean_2nd", "SEM_2nd",
                     "MeanDiff_2nd_minus_1st", "t", "p_two_sided", "Cohen_dz"])
    for g in ("Increase", "Decrease"):
        rs = [r for r in results if r["Group"] == g]
        for kind, w0, w1 in PEAK_WINDOWS[g]:
            metric = f"{kind}_{w0}_{w1}s"
            v1 = np.array([r["peaks"][metric][0] for r in rs])
            v2 = np.array([r["peaks"][metric][1] for r in rs])
            diff = v2 - v1
            res = stats.ttest_rel(v2, v1)
            writer.writerow([g, metric, len(rs),
                             v1.mean(), v1.std(ddof=1) / np.sqrt(len(v1)),
                             v2.mean(), v2.std(ddof=1) / np.sqrt(len(v2)),
                             diff.mean(), res.statistic, res.pvalue,
                             diff.mean() / diff.std(ddof=1)])
            print(f"{g} {metric}: n={len(rs)}, p={res.pvalue:.4f}, dz={diff.mean() / diff.std(ddof=1):.3f}")

# (B-left) Figure S8B: mean ± SEM trace
#   마우스별 trace에 0.5 s 이동평균(centered) 적용 후 mean ± SEM 계산 (표시용)
#   CSV에는 스무딩 전(Raw)/후(Smoothed) 값을 모두 저장
SMOOTH_N = max(1, int(round(SMOOTH_S / TRACE_STEP)))   # 0.5 s / 0.05 s = 10 points

def smooth_traces(matrix):
    """각 마우스 trace(행)에 centered 이동평균 적용"""
    return (pd.DataFrame(matrix.T)
            .rolling(window=SMOOTH_N, center=True, min_periods=1)
            .mean()
            .to_numpy().T)

def mean_sem(matrix):
    n = np.sum(np.isfinite(matrix), axis=0)
    return np.nanmean(matrix, axis=0), np.nanstd(matrix, axis=0, ddof=1) / np.sqrt(n)

trace_raw, trace_smooth = {}, {}
for g in ("Increase", "Decrease"):
    for s in ("1st", "2nd"):
        mat = np.array([r["traces"][s] for r in results if r["Group"] == g])
        trace_raw[(g, s)]    = mean_sem(mat)
        trace_smooth[(g, s)] = mean_sem(smooth_traces(mat))

out_trace = os.path.join(save_dir, "FigS8B_mean_traces.csv")
with open(out_trace, "w", newline="") as f:
    writer = csv.writer(f)
    keys = list(trace_raw.keys())
    header = ["Time_s"]
    for g, s in keys:
        header += [f"{g}_{s}_Mean_Smoothed", f"{g}_{s}_SEM_Smoothed",
                   f"{g}_{s}_Mean_Raw", f"{g}_{s}_SEM_Raw"]
    writer.writerow(header)
    for i, t in enumerate(TRACE_TIME):
        row = [round(t, 3)]
        for k in keys:
            row += [trace_smooth[k][0][i], trace_smooth[k][1][i],
                    trace_raw[k][0][i], trace_raw[k][1][i]]
        writer.writerow(row)

print(f"\nCSV 저장 완료:\n  {out_A}\n  {out_peak}\n  {out_stat}\n  {out_trace}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 4) Figure S8B: mean ± SEM trace (1st 검정, 2nd 색)
# ──────────────────────────────────────────────────────────────────────────────
PANELS = [
    ("Increase", "Immediate increase", "#E41A1C"),
    ("Decrease", "Immediate decrease followed by recovery", "#1FA3E0"),
]

fig, axes = plt.subplots(2, 1, figsize=(6, 7), sharey=True)
for ax, (g, title, color) in zip(axes, PANELS):
    for s, c in (("1st", "black"), ("2nd", color)):
        m, sem = trace_smooth[(g, s)]
        ax.plot(TRACE_TIME, m, color=c, lw=1.2, label=s)
        ax.fill_between(TRACE_TIME, m - sem, m + sem, color=c, alpha=0.2, lw=0)
    ax.axvspan(0, 3, color="#80CDC1", alpha=0.2, lw=0)   # tail pinch 0-3 s
    ax.axvline(0, color="k", ls="--", lw=0.8)
    ax.axvline(3, color="k", ls=":", lw=0.8)
    ax.set_xlim(TRACE_START, TRACE_END)
    ax.set_xticks([-60, -30, 0, 30, 60, 90, 120])
    ax.set_ylim(-5.8, 4.6)
    ax.set_yticks([-4, -2, 0, 2, 4])
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Z-score")
    ax.set_title(title, fontsize=10)
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(save_dir, "FigS8B_mean_traces.png"), dpi=300)
plt.savefig(os.path.join(save_dir, "FigS8B_mean_traces.svg"))
plt.show()